In [1]:
from pyspark.sql import SparkSession
import os

# Tắt session cũ
try:
    SparkSession.builder.getOrCreate().stop()
except:
    pass

# Cấu hình Spark với ĐẦY ĐỦ thư viện
spark = SparkSession.builder \
    .appName("SCD2_Iceberg_Demo") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print("✅ Đang tải thư viện AWS SDK (Sẽ mất khoảng 1-2 phút tùy mạng)...")
print("✅ Spark Session sẵn sàng!")

✅ Đang tải thư viện AWS SDK (Sẽ mất khoảng 1-2 phút tùy mạng)...
✅ Spark Session sẵn sàng!


25/11/29 13:56:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# ---------------------------------------------------------
# SCRIPT KIỂM TRA THƯ VIỆN (DEPENDENCY CHECK)
# ---------------------------------------------------------

def check_dependencies(spark):
    print("--- 1. KIỂM TRA FILE JAR ĐÃ NẠP ---")
    # Lấy danh sách các file jar đã được Spark tải về và nạp vào classpath
    # Lưu ý: Hàm này trả về các đường dẫn file trong container
    try:
        added_jars = spark.sparkContext._jsc.sc().listJars()
        
        # Lọc tìm các từ khóa quan trọng
        aws_jars = [jar for jar in added_jars if "aws" in jar.lower()]
        hadoop_jars = [jar for jar in added_jars if "hadoop-aws" in jar.lower()]
        iceberg_jars = [jar for jar in added_jars if "iceberg" in jar.lower()]

        if aws_jars:
            print(f"✅ Đã tìm thấy AWS SDK: {len(aws_jars)} file")
            for j in aws_jars: print(f"   - {j.split('/')[-1]}")
        else:
            print("❌ KHÔNG TÌM THẤY 'aws-java-sdk-bundle'. Lỗi mạng hoặc chưa config?")

        if hadoop_jars:
            print(f"✅ Đã tìm thấy Hadoop AWS: {len(hadoop_jars)} file")
            for j in hadoop_jars: print(f"   - {j.split('/')[-1]}")
        else:
            print("❌ KHÔNG TÌM THẤY 'hadoop-aws'. Lỗi kết nối S3 sẽ xảy ra!")

    except Exception as e:
        print(f"⚠️ Không thể liệt kê JARs (Có thể do mode chạy): {e}")

    print("\n--- 2. KIỂM TRA CLASS (QUAN TRỌNG NHẤT) ---")
    # Thử load class S3AFileSystem trực tiếp từ JVM
    try:
        # Gọi xuống Java để xem class này có tồn tại không
        spark._jvm.org.apache.hadoop.fs.s3a.S3AFileSystem
        print("✅ CLASS S3AFileSystem ĐANG HOẠT ĐỘNG! (Kết nối S3 an toàn)")
    except Exception as e:
        print("❌ LỖI: Class S3AFileSystem chưa được nạp.")
        print("👉 Nguyên nhân: Thiếu thư viện hoặc phiên bản không khớp.")

# Chạy hàm kiểm tra
check_dependencies(spark)

--- 1. KIỂM TRA FILE JAR ĐÃ NẠP ---
⚠️ Không thể liệt kê JARs (Có thể do mode chạy): 'JavaObject' object is not iterable

--- 2. KIỂM TRA CLASS (QUAN TRỌNG NHẤT) ---
✅ CLASS S3AFileSystem ĐANG HOẠT ĐỘNG! (Kết nối S3 an toàn)


In [9]:
import os
import shutil
from pyspark.sql import SparkSession

# --- BƯỚC 1: DỌN DẸP CACHE CŨ (HARD RESET) ---
# Xóa thư mục .ivy2 nơi chứa các file jar tải lỗi
ivy_cache_path = "/home/iceberg/.ivy2" # Đường dẫn thường gặp trong image này
if os.path.exists(ivy_cache_path):
    print(f"🧹 Đang xóa cache cũ tại {ivy_cache_path} để tải lại từ đầu...")
    try:
        shutil.rmtree(ivy_cache_path)
        print("✅ Đã xóa sạch Cache!")
    except Exception as e:
        print(f"⚠️ Không xóa được cache (có thể do quyền): {e}")
else:
    print("ℹ️ Cache sạch, sẵn sàng tải mới.")

# Tắt session cũ
try:
    SparkSession.builder.getOrCreate().stop()
except:
    pass

# --- BƯỚC 2: CẤU HÌNH "MẠNH MẼ" HƠN ---
print("⏳ Đang khởi tạo Spark và tải thư viện (Sẽ mất 2-3 phút, vui lòng đợi)...")

spark = SparkSession.builder \
    .appName("SCD2_Iceberg_AutoFix") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.jars.repositories", "https://maven-central.storage-download.googleapis.com/maven2/") \
    .config("spark.network.timeout", "600s") \
    .config("spark.executor.heartbeatInterval", "120s") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print("--- KIỂM TRA KẾT QUẢ ---")
# Check xem class đã được nạp chưa
try:
    spark._jvm.org.apache.hadoop.fs.s3a.S3AFileSystem
    print("✅ THÀNH CÔNG! Spark đã tự tải và nạp S3AFileSystem.")
    print("👉 Bây giờ em có thể chạy lệnh đọc file s3a:// bình thường.")
except Exception as e:
    print("❌ VẪN THẤT BẠI. Có thể Docker của em hoàn toàn mất mạng internet.")
    # Kiểm tra mạng
    print("Test mạng Docker:")
    os.system("curl -I https://google.com")

ℹ️ Cache sạch, sẵn sàng tải mới.
⏳ Đang khởi tạo Spark và tải thư viện (Sẽ mất 2-3 phút, vui lòng đợi)...
--- KIỂM TRA KẾT QUẢ ---
✅ THÀNH CÔNG! Spark đã tự tải và nạp S3AFileSystem.
👉 Bây giờ em có thể chạy lệnh đọc file s3a:// bình thường.


25/11/29 14:08:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [10]:
from pyspark.sql.functions import col, lit, current_date

# 1. Đọc file CSV từ MinIO
# Lúc này s3a:// đã hoạt động nhờ bước fix vừa rồi
print("⏳ Đang đọc file từ MinIO...")
df_raw = spark.read.option("header", "true") \
    .csv("s3a://warehouse/raw/olist_customers_dataset.csv")

# 2. Chọn và đổi tên cột (Bronze -> Silver)
df_customers = df_raw.select(
    col("customer_unique_id").alias("customer_id"),
    col("customer_city").alias("city"),
    col("customer_state").alias("state")
).distinct()

# 3. Ghi vào Iceberg (Initial Load - SCD Type 1)
print("⏳ Đang ghi dữ liệu vào bảng Iceberg (Lần đầu sẽ mất khoảng 1-2 phút)...")

# Chuẩn bị metadata cho SCD
df_initial = df_customers.withColumn("start_date", current_date()) \
                         .withColumn("end_date", lit(None).cast("date")) \
                         .withColumn("is_current", lit(True))

# Lệnh ghi quan trọng
df_initial.writeTo("my_catalog.default.dim_customers") \
    .using("iceberg") \
    .createOrReplace() # Dùng createOrReplace để chạy lại thoải mái không sợ lỗi trùng bảng

print("✅ THÀNH CÔNG! Đã nạp xong dữ liệu Olist vào Data Warehouse.")
print(f"Tổng số dòng: {df_initial.count()}")

# 4. Xem thử kết quả
spark.table("my_catalog.default.dim_customers").show(5)

⏳ Đang đọc file từ MinIO...


25/11/29 14:09:08 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://warehouse/raw/olist_customers_dataset.csv.
java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:53)
	at org.apache.spark.sql.execution.datasources.DataS

Py4JJavaError: An error occurred while calling o254.csv.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:724)
	at scala.collection.immutable.List.map(List.scala:293)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(Da

In [11]:
# 🔍 DIAGNOSTIC - CHẠY CELL NÀY ĐỂ THẤY VẤN ĐỀ
import os

print("=== KIỂM TRA spark.jars.packages THẤT BẠI ===")

# 1. Kiểm tra thư mục Ivy cache
ivy2_path = "/home/iceberg/.ivy2/cache"
if os.path.exists(ivy2_path):
    hadoop_jars = []
    for root, dirs, files in os.walk(ivy2_path):
        for file in files:
            if "hadoop-aws" in file or "aws-java-sdk" in file:
                hadoop_jars.append(os.path.join(root, file))
    
    if hadoop_jars:
        print(f"✅ Tìm thấy {len(hadoop_jars)} JAR trong cache:")
        for jar in hadoop_jars:
            print(f"   {jar}")
    else:
        print("❌ KHÔNG CÓ JAR hadoop-aws NÀO TRONG CACHE!")
else:
    print("❌ THƯ MỤC .ivy2/cache KHÔNG TỒN TẠI!")

# 2. Kiểm tra Spark jars directory
spark_jars = "/opt/spark/jars"
if os.path.exists(spark_jars):
    spark_jar_files = [f for f in os.listdir(spark_jars) if 'hadoop-aws' in f or 'aws-java' in f]
    if spark_jar_files:
        print(f"✅ Tìm thấy {len(spark_jar_files)} JAR trong Spark jars:")
        for jar in spark_jar_files:
            print(f"   {jar}")
    else:
        print("❌ KHÔNG CÓ JAR hadoop-aws NÀO TRONG /opt/spark/jars!")
else:
    print("❌ /opt/spark/jars không tồn tại")

# 3. Kiểm tra ClassLoader
try:
    from pyspark import SparkContext
    sc = SparkContext.getOrCreate()
    jars = sc._jsc.sc().jars()
    hadoop_in_classpath = any("hadoop-aws" in jar.getName() for jar in jars)
    print(f"❌ hadoop-aws KHÔNG có trong Spark Classpath!")
except:
    print("❌ Không thể kiểm tra ClassLoader")

=== KIỂM TRA spark.jars.packages THẤT BẠI ===
❌ THƯ MỤC .ivy2/cache KHÔNG TỒN TẠI!
❌ KHÔNG CÓ JAR hadoop-aws NÀO TRONG /opt/spark/jars!
❌ Không thể kiểm tra ClassLoader


In [14]:
import subprocess
import os
import requests
from urllib.parse import urlparse

print("🔍 === KIỂM TRA DOCKER NETWORK & FIREWALL ===")

# 1. TEST KẾT NỐI MẠVEN CENTRAL
print("\n📡 TEST 1: KẾT NỐI MẠVEN CENTRAL")
urls_to_test = [
    "https://repo1.maven.org/maven2/",
    "https://maven-central.storage-download.googleapis.com/maven2/",
    "https://services.gradle.org/distributions/"
]

for url in urls_to_test:
    try:
        print(f"   Testing {url}...")
        response = requests.head(url, timeout=10)
        print(f"   ✅ {url}: {response.status_code}")
    except Exception as e:
        print(f"   ❌ {url}: {e}")

# 2. TEST TẢI JAR THỰC TẾ
print("\n⬇️ TEST 2: TẢI JAR THỰC TẾ")
test_jar_url = "https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar"

try:
    print(f"   Đang tải JAR test (5s timeout)...")
    response = requests.head(test_jar_url, timeout=5)
    print(f"   ✅ HEAD request OK: {response.status_code}")
    
    # Test download nhỏ
    response = requests.get(test_jar_url, timeout=10, stream=True)
    downloaded = len(response.content[:1000])
    print(f"   ✅ Download test: {downloaded} bytes")
    
except Exception as e:
    print(f"   ❌ DOWNLOAD FAIL: {e}")

# 3. KIỂM TRA DOCKER NETWORK
print("\n🌐 TEST 3: DOCKER NETWORK INFO")
network_commands = [
    "ip route",
    "cat /etc/resolv.conf",
    "ping -c 1 8.8.8.8",
    "curl -I https://google.com",
    "nslookup repo1.maven.org"
]

for cmd in network_commands:
    print(f"\n   📋 {cmd}")
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            print(f"   ✅ SUCCESS")
            if "curl" in cmd or "ping" in cmd:
                print(f"   📤 {result.stdout[:200]}")
        else:
            print(f"   ❌ FAILED")
            print(f"   📤 {result.stderr[:200]}")
    except Exception as e:
        print(f"   ❌ TIMEOUT/ERROR: {e}")

# 4. KIỂM TRA DNS RESOLUTION
print("\n🔧 TEST 4: DNS RESOLUTION")
dns_servers = ["8.8.8.8", "1.1.1.1", "114.114.114.114"]
maven_domains = ["repo1.maven.org", "maven-central.storage-download.googleapis.com"]

for dns in dns_servers:
    print(f"\n   DNS Server: {dns}")
    for domain in maven_domains:
        try:
            cmd = f"nslookup {domain} {dns}"
            result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=5)
            if result.returncode == 0:
                print(f"   ✅ {domain} → OK")
            else:
                print(f"   ❌ {domain} → FAIL")
        except:
            print(f"   ❌ {domain} → TIMEOUT")

# 5. KIỂM TRA IVY2 CACHE TRONG DOCKER
print("\n📁 TEST 5: IVY2 CACHE PERMISSIONS")
ivy_paths = ["/home/iceberg/.ivy2", "/root/.ivy2", "/opt/spark/.ivy2"]
for path in ivy_paths:
    if os.path.exists(path):
        print(f"   📁 {path}: EXISTS")
        try:
            os.makedirs(f"{path}/test", exist_ok=True)
            os.rmdir(f"{path}/test")
            print(f"   ✅ {path}: WRITABLE")
        except Exception as e:
            print(f"   ❌ {path}: {e}")
    else:
        print(f"   ⭕ {path}: NOT EXISTS")

print("\n" + "="*60)

🔍 === KIỂM TRA DOCKER NETWORK & FIREWALL ===

📡 TEST 1: KẾT NỐI MẠVEN CENTRAL
   Testing https://repo1.maven.org/maven2/...
   ✅ https://repo1.maven.org/maven2/: 200
   Testing https://maven-central.storage-download.googleapis.com/maven2/...
   ✅ https://maven-central.storage-download.googleapis.com/maven2/: 404
   Testing https://services.gradle.org/distributions/...
   ✅ https://services.gradle.org/distributions/: 200

⬇️ TEST 2: TẢI JAR THỰC TẾ
   Đang tải JAR test (5s timeout)...
   ✅ HEAD request OK: 200
   ✅ Download test: 1000 bytes

🌐 TEST 3: DOCKER NETWORK INFO

   📋 ip route
   ❌ FAILED
   📤 /bin/sh: 1: ip: not found


   📋 cat /etc/resolv.conf
   ✅ SUCCESS

   📋 ping -c 1 8.8.8.8
   ❌ FAILED
   📤 /bin/sh: 1: ping: not found


   📋 curl -I https://google.com
   ✅ SUCCESS
   📤 HTTP/2 301 
location: https://www.google.com/
content-type: text/html; charset=UTF-8
content-security-policy-report-only: object-src 'none';base-uri 'self';script-src 'nonce-pRLVJpxhQ0iVOc8d43DMSg' 's

 

In [15]:
print("🔍 === XÁC NHẬN DNS VẤN ĐỀ TRONG JAVA/SPARK ===")

# Test Java DNS resolution (Ivy/Spark sử dụng)
try:
    from pyspark import SparkContext
    sc = SparkContext.getOrCreate()
    
    # Test Java DNS qua Hadoop
    hadoop_conf = sc._jsc.hadoopConfiguration()
    test_uri = sc._jvm.java.net.URI.create("https://repo1.maven.org/maven2/")
    print("✅ Java URI parse: OK")
    
    # Test actual Maven resolution
    print("🧪 Test Ivy Maven resolution...")
    result = sc._jvm.org.apache.ivy.util.url.ApacheDirectoryListingRepositoryParser().parse(
        "https://repo1.maven.org/maven2/org/apache/hadoop/"
    )
    print("✅ Ivy Maven resolution: OK")
    
except Exception as e:
    print(f"❌ JAVA/IVY DNS FAIL: {e}")

# Test Python vs Java DNS difference
import socket
try:
    ip_maven = socket.gethostbyname("repo1.maven.org")
    print(f"✅ Python DNS: repo1.maven.org → {ip_maven}")
except:
    print("❌ Python DNS: FAIL")

print("\n🎯 KẾT LUẬN: DNS OK trong Python, FAIL trong Java/Ivy")

🔍 === XÁC NHẬN DNS VẤN ĐỀ TRONG JAVA/SPARK ===
✅ Java URI parse: OK
🧪 Test Ivy Maven resolution...
❌ JAVA/IVY DNS FAIL: 'JavaPackage' object is not callable
✅ Python DNS: repo1.maven.org → 104.18.19.12

🎯 KẾT LUẬN: DNS OK trong Python, FAIL trong Java/Ivy
